# Dynamo Models of the Solar Cycle (Charbonneau 2020) — Implementation / 구현

**Paper**: Charbonneau, P., *Living Reviews in Solar Physics*, **17**, 4 (2020). DOI: 10.1007/s41116-020-00025-6

This notebook implements the key algorithms and concepts from Charbonneau (2020):
1. Kinematic α-Ω dynamo wave (1D Cartesian Parker model)
2. Babcock–Leighton iterated map with stochastic AR tilt
3. Flux-transport dynamo schematic (2D meridional plane)
4. Grand Minima occurrence from the stochastic BL model
5. Butterfly diagram from the BL iterated map
6. Cycle period vs. α amplitude

이 노트북은 Charbonneau(2020)의 핵심 알고리즘을 구현합니다:
1. Kinematic α-Ω 다이나모 파동(1D Parker 모델)
2. 확률적 AR tilt를 포함한 Babcock–Leighton iterated map
3. Flux-transport 다이나모 개략(2D 자오면)
4. 확률적 BL 모델의 Grand Minima 발생
5. BL 모델의 나비 다이어그램
6. 주기 vs. α 진폭

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
from scipy.special import erf

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
rng = default_rng(seed=42)

## Part 1: Kinematic α-Ω dynamo wave (1D Parker) / Kinematic α-Ω 다이나모 파동

We solve the classical Parker (1955) dynamo equations in 1D Cartesian geometry, illustrating the Parker–Yoshimura sign rule and dynamo-wave propagation.

Equations (Sect. 4.2.7 reduced to 1D, with $y$ = latitude):

$$\frac{\partial A}{\partial t}=\alpha B+\eta\frac{\partial^2 A}{\partial y^2}$$

$$\frac{\partial B}{\partial t}=-G\frac{\partial A}{\partial y}+\eta\frac{\partial^2 B}{\partial y^2}$$

with $G=\partial\Omega/\partial y$ the latitudinal shear. Parker–Yoshimura sign rule: waves travel as $\mathbf{s}=\alpha\nabla\Omega\times\hat{\mathbf{e}}_\phi$.

Parker(1955) 1D 다이나모 방정식의 유한차분 수치해법. α>0, G<0 → 파동은 +y 방향(북반구 기준 equatorward).

In [ ]:
def dynamo_wave_1d(alpha=1.0, G=-1.0, eta=0.05, Ny=201, Ly=np.pi, T=40.0, dt=0.005):
    """Integrate the 1D Parker dynamo wave equations with explicit finite differences.

    Args:
        alpha: Amplitude of the alpha-effect (positive by convention).
        G: Latitudinal shear dOmega/dy. Sign controls wave direction.
        eta: Turbulent magnetic diffusivity.
        Ny: Number of grid points in y.
        Ly: Domain length in y.
        T: Total integration time.
        dt: Time step.

    Returns:
        Tuple of (y, t_arr, A_hist, B_hist) arrays.
    """
    y = np.linspace(0, Ly, Ny)
    dy = y[1] - y[0]
    A = 1e-3 * np.sin(y)
    B = np.zeros_like(A)
    nt = int(T / dt)
    save_every = max(1, nt // 400)
    A_hist, B_hist, t_arr = [], [], []
    for it in range(nt):
        d2A = np.zeros_like(A)
        d2B = np.zeros_like(B)
        d2A[1:-1] = (A[2:] - 2*A[1:-1] + A[:-2]) / dy**2
        d2B[1:-1] = (B[2:] - 2*B[1:-1] + B[:-2]) / dy**2
        dAdy = np.zeros_like(A)
        dAdy[1:-1] = (A[2:] - A[:-2]) / (2*dy)
        sat = 1.0 / (1.0 + (B / 1.0)**2)
        A += dt * (alpha * sat * B + eta * d2A)
        B += dt * (-G * dAdy + eta * d2B)
        A[0] = A[-1] = 0.0
        B[0] = B[-1] = 0.0
        if it % save_every == 0:
            A_hist.append(A.copy())
            B_hist.append(B.copy())
            t_arr.append(it*dt)
    return y, np.array(t_arr), np.array(A_hist), np.array(B_hist)

y, t, A_hist, B_hist = dynamo_wave_1d(alpha=2.0, G=-5.0, eta=0.05, T=15.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
im0 = axes[0].imshow(A_hist.T, aspect='auto', origin='lower',
                     extent=[t[0], t[-1], y[0], y[-1]], cmap='RdBu_r')
axes[0].set_title('Poloidal potential A(y,t)')
axes[0].set_xlabel('time'); axes[0].set_ylabel('y (latitude proxy)')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(B_hist.T, aspect='auto', origin='lower',
                     extent=[t[0], t[-1], y[0], y[-1]], cmap='RdBu_r')
axes[1].set_title('Toroidal field B(y,t) — dynamo wave')
axes[1].set_xlabel('time'); axes[1].set_ylabel('y (latitude proxy)')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

period_est = 2*np.pi/np.sqrt(abs(2.0*(-5.0))/2.0)
print(f'Rough Parker period estimate 2π/sqrt(|αG|/2) = {period_est:.3f}')

**Observation / 관찰**: The toroidal field $B$ drifts systematically in the $+y$ direction for $\alpha>0$, $G<0$ (Parker–Yoshimura). Interpreting $+y$ as equatorward reproduces the sunspot butterfly.

북반구 양의 α와 음의 shear 조합이 equatorward 다이나모 파를 생성—태양 butterfly의 설명 하나.

## Part 2: Babcock–Leighton iterated map with stochastic tilt / 확률적 tilt BL iterated map

Durney (2000) / Charbonneau & Dikpati (2000)-style iterated map:

$$P_{n+1}=s\cdot f(T_n)\cdot(1+\xi_n),\qquad T_{n+1}=g\cdot P_{n+1}$$

with tilt quenching $f(T)=T/(1+(T/T_0)^2)$ and multiplicative Gaussian noise $\xi_n\sim\mathcal{N}(0,\sigma^2)$ matching the ~15° Joy's-law scatter.

Cameron & Schussler(2017b) 확률 Hopf SDE의 이산 아날로그. Multiplicative noise.

In [ ]:
def bl_iterated_map(Ncyc=1000, s=1.0, g=1.0, T0=1.0, sigma=0.3, threshold=0.2, seed=1):
    """Stochastic Babcock-Leighton iterated map with tilt quenching and lower threshold.

    Args:
        Ncyc: Number of cycles.
        s: Babcock-Leighton T->P conversion efficiency.
        g: Omega-effect P->T conversion efficiency.
        T0: Tilt-quenching scale.
        sigma: Stochastic tilt scatter (fractional).
        threshold: Lower operating threshold on toroidal field.
        seed: RNG seed.

    Returns:
        Tuple of (P_arr, T_arr) arrays.
    """
    local_rng = default_rng(seed)
    P = np.zeros(Ncyc)
    T = np.zeros(Ncyc)
    P[0] = 1.0
    T[0] = g * P[0]
    for n in range(1, Ncyc):
        if abs(T[n-1]) < threshold:
            P[n] = 0.05 * local_rng.normal() + P[n-1] * 0.5
        else:
            xi = local_rng.normal()
            P[n] = s * T[n-1] / (1 + (T[n-1]/T0)**2) * (1 + sigma*xi)
        T[n] = g * P[n]
    return P, T

P_arr, T_arr = bl_iterated_map(Ncyc=500, s=1.4, g=1.0, T0=1.0, sigma=0.25, threshold=0.2, seed=3)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(np.abs(P_arr), color='tab:blue')
axes[0].set_ylabel('|P| (dipole)')
axes[0].set_title('Stochastic BL iterated map — dipole amplitude')
axes[1].plot(np.abs(T_arr), color='tab:red')
axes[1].set_ylabel('|T| (toroidal)')
axes[1].set_xlabel('cycle number n')
axes[1].axhline(0.2, color='k', ls='--', alpha=0.5, label='threshold')
axes[1].legend()
plt.tight_layout()
plt.show()

## Part 3: Flux-transport dynamo schematic (2D meridional plane) / Flux-transport 개략

Schematic snapshot of a BL flux-transport dynamo in the meridional plane:
- Differential rotation $\Omega(r,\theta)$ (paper Eq. 20).
- Single-cell meridional circulation streamlines (van Ballegooijen & Choudhuri 1988).

논문 Fig. 4의 개략적 복제. 자오면의 differential rotation과 single-cell meridional circulation.

In [ ]:
Nr, Nt = 120, 120
r = np.linspace(0.6, 1.0, Nr)
theta = np.linspace(0.0, np.pi/2, Nt)
R, TH = np.meshgrid(r, theta, indexing='ij')

OmS = 1.0 - 0.1264*np.cos(TH)**2 - 0.1591*np.cos(TH)**4
Omega = 0.8752 + 0.5*(OmS - 0.8752)*(1.0 + erf((R - 0.7)/0.05))

rb = 0.675
r_s = np.clip((R - rb)/(1.0 - rb), 1e-6, 1.0)
PSI = np.where(R > rb, r_s**0.5 * (1.0 - r_s) * np.sin(TH)**2 * np.cos(TH), 0.0)

X = R * np.sin(TH)
Y = R * np.cos(TH)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
cs0 = axes[0].contour(X, Y, Omega, levels=20, cmap='viridis')
axes[0].plot(0.7*np.sin(theta), 0.7*np.cos(theta), 'k--', lw=1, label='tachocline r=0.7')
axes[0].set_aspect('equal')
axes[0].set_title(r'Differential rotation $\Omega(r,\theta)/\Omega_E$')
axes[0].set_xlabel('equatorial')
axes[0].set_ylabel('polar axis')
axes[0].legend()
plt.colorbar(cs0, ax=axes[0])

cs1 = axes[1].contour(X, Y, PSI, levels=15, cmap='RdBu_r')
axes[1].plot(0.7*np.sin(theta), 0.7*np.cos(theta), 'k--', lw=1)
axes[1].set_aspect('equal')
axes[1].set_title('Meridional circulation streamlines (single cell)')
axes[1].set_xlabel('equatorial')
axes[1].set_ylabel('polar axis')
plt.colorbar(cs1, ax=axes[1])
plt.tight_layout()
plt.show()

## Part 4: Grand Minima occurrence from stochastic BL model / 확률적 BL 모델의 Grand Minima

Using the stochastic iterated map from Part 2, we count Grand Minima — contiguous runs of ≥3 cycles with $|T_n|<$ 30% of the mean. We expect an approximately exponential distribution of inter-event waiting times (memoryless stochastic process), matching cosmogenic radioisotope records (Usoskin 2017).

Part 2의 iterated map에서 Grand Minima(3 cycle 이상 평균 대비 30% 미만) 발생을 집계. Inter-event waiting time이 대략 exponential 분포.

In [ ]:
def detect_grand_minima(T_arr, thresh_frac=0.3, min_len=3):
    """Detect Grand Minima events and return their start indices and durations.

    Args:
        T_arr: Cycle-by-cycle toroidal amplitudes.
        thresh_frac: Threshold as a fraction of the mean |T|.
        min_len: Minimum number of contiguous sub-threshold cycles to count.

    Returns:
        List of (start_index, duration) tuples.
    """
    mag = np.abs(T_arr)
    thr = thresh_frac * np.mean(mag)
    in_min = mag < thr
    events = []
    start = None
    for i, flag in enumerate(in_min):
        if flag and start is None:
            start = i
        elif (not flag) and start is not None:
            if i - start >= min_len:
                events.append((start, i - start))
            start = None
    if start is not None and len(T_arr) - start >= min_len:
        events.append((start, len(T_arr) - start))
    return events

P_long, T_long = bl_iterated_map(Ncyc=5000, s=1.4, g=1.0, T0=1.0, sigma=0.3, threshold=0.2, seed=42)
events = detect_grand_minima(T_long, thresh_frac=0.3, min_len=3)
starts = np.array([e[0] for e in events])
waiting = np.diff(starts) if len(starts) > 1 else np.array([])
durations = np.array([e[1] for e in events])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(np.abs(T_long), lw=0.6, color='tab:red')
for s_, d_ in events[:30]:
    axes[0].axvspan(s_, s_+d_, color='gray', alpha=0.3)
axes[0].set_title(f'Grand Minima detected: {len(events)} events over 5000 cycles')
axes[0].set_xlabel('cycle number')
axes[0].set_ylabel('|T|')
if len(waiting) > 5:
    axes[1].hist(waiting, bins=30, color='tab:blue', alpha=0.7, density=True, label='simulated')
    lam = 1.0 / np.mean(waiting)
    wx = np.linspace(0, waiting.max(), 200)
    axes[1].plot(wx, lam*np.exp(-lam*wx), 'k--', label=f'exp fit, 1/λ = {1/lam:.1f} cyc')
    axes[1].set_title('Inter-event waiting time distribution')
    axes[1].set_xlabel('waiting time (cycles)')
    axes[1].legend()
plt.tight_layout()
plt.show()
if len(waiting) > 0:
    print(f'Mean waiting time: {np.mean(waiting):.1f} cycles; mean duration: {np.mean(durations):.1f} cycles')

## Part 5: Butterfly diagram from BL model / BL 모델 나비 다이어그램

Synthetic butterfly diagram by advecting toroidal-field source with a latitudinally structured pattern: equatorward drift of maxima over each cycle.

토로이달 자기장의 latitude 구조를 간단 advection+source 모델로 구성. 각 주기 안에서 equatorward drift가 나타남.

In [ ]:
def butterfly_from_bl(Nlat=80, Ncyc=10, cycle_period=11.0, dt=0.05):
    """Generate a schematic butterfly diagram with equatorward peak drift.

    The amplitude at each latitude evolves as a harmonic modulation whose
    peak latitude sweeps from 35 to 5 degrees over each cycle.

    Args:
        Nlat: Latitude grid points (hemisphere).
        Ncyc: Number of cycles to simulate.
        cycle_period: Cycle period in years.
        dt: Time step in years.

    Returns:
        Tuple of (t_arr, lat_arr, B_lat_t) arrays.
    """
    lat = np.linspace(0, 60, Nlat)
    T_total = cycle_period * Ncyc
    nt = int(T_total / dt)
    t_arr = np.linspace(0, T_total, nt)
    B = np.zeros((nt, Nlat))
    for i, ti in enumerate(t_arr):
        phase = 2*np.pi * ti / cycle_period
        frac = (ti % cycle_period) / cycle_period
        peak_lat = 35 - 30 * frac
        amp = np.sin(phase)
        gauss = np.exp(-((lat - peak_lat)/8.0)**2)
        B[i, :] = amp * gauss
    return t_arr, lat, B

t_arr, lat_arr, B_lat_t = butterfly_from_bl(Nlat=100, Ncyc=8, cycle_period=11.0, dt=0.05)
fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(B_lat_t.T, aspect='auto', origin='lower',
               extent=[t_arr[0], t_arr[-1], lat_arr[0], lat_arr[-1]],
               cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('time (years)')
ax.set_ylabel('latitude (deg)')
ax.set_title('Synthetic butterfly diagram (schematic)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Part 6: Cycle period vs. α amplitude / 주기 vs. α 진폭

In the αΩ dynamo, linear-regime cycle frequency increases with dynamo number. In the saturated regime, it becomes less sensitive to $D$ and scales primarily with the magnetic diffusion time $\tau=R^2/\eta_T$. We sweep $\alpha$ and measure the period.

αΩ 다이나모에서 linear 영역의 주기 주파수는 dynamo number에 따라 증가. 포화 상태에서는 D에 덜 민감.

In [ ]:
def cycle_period_from_1d(alpha, G=-5.0, eta=0.05, T=30.0, dt=0.005, Ny=101, Ly=np.pi):
    """Measure the dominant cycle period of the 1D Parker dynamo via zero-crossings.

    Args:
        alpha: Alpha-effect amplitude.
        G: Latitudinal shear.
        eta: Turbulent diffusivity.
        T: Total integration time.
        dt: Time step.
        Ny: Number of grid points.
        Ly: Domain length.

    Returns:
        Mean period (float) estimated from the second half of the run.
    """
    y = np.linspace(0, Ly, Ny)
    dy = y[1] - y[0]
    A = 1e-3 * np.sin(y)
    B = np.zeros_like(A)
    nt = int(T / dt)
    B_center = []
    for it in range(nt):
        d2A = np.zeros_like(A)
        d2B = np.zeros_like(B)
        d2A[1:-1] = (A[2:] - 2*A[1:-1] + A[:-2]) / dy**2
        d2B[1:-1] = (B[2:] - 2*B[1:-1] + B[:-2]) / dy**2
        dAdy = np.zeros_like(A)
        dAdy[1:-1] = (A[2:] - A[:-2]) / (2*dy)
        sat = 1.0 / (1.0 + (B/1.0)**2)
        A += dt * (alpha * sat * B + eta * d2A)
        B += dt * (-G * dAdy + eta * d2B)
        A[0] = A[-1] = 0.0
        B[0] = B[-1] = 0.0
        B_center.append(B[Ny//2])
    B_center = np.array(B_center)
    sig = B_center[nt//2:]
    zc = np.where(np.diff(np.sign(sig)))[0]
    if len(zc) < 4:
        return np.nan
    periods = 2 * np.diff(zc) * dt
    return float(np.mean(periods))

alpha_scan = np.linspace(1.0, 6.0, 12)
periods = np.array([cycle_period_from_1d(a, T=25.0, Ny=81) for a in alpha_scan])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alpha_scan, periods, 'o-', color='tab:purple')
ax.set_xlabel(r'$\alpha$ amplitude')
ax.set_ylabel('cycle period (arb. units)')
ax.set_title('Cycle period vs. α amplitude (saturated 1D Parker dynamo)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Periods:', [f'{p:.2f}' for p in periods])

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| α-Ω dynamo wave | Parker (1955) 1D Cartesian | 2D axisymmetric αΩ in spherical shells (Sect. 4.2) |
| Babcock–Leighton source | Joy's-law-tilted BMR decay (Sect. 5.1, Eq. 49) | 2×2D coupled SFT + interior dynamo (Lemerle & Charbonneau 2017) |
| Flux-transport dynamo | Meridional-circulation-paced cycle (Sect. 4.4, 5.3) | Helioseismic data-assimilated BL (Hung et al. 2017) |
| Stochastic Hopf SDE | Cameron & Schüssler 2017b (Eq. 64) | Stochastic 2×2D BL with observed BMR statistics |
| Grand Minima | On–off / in–out intermittency (Sect. 7.3) | Augustson et al. 2015 MHD K3S simulation |
| Butterfly diagram | Dynamo wave + Parker–Yoshimura (Sect. 4.2.9) | 3D BL with tilt quenching (Karak & Miesch 2017) |
| Cycle prediction | Surface dipole as precursor (Sect. 5.6) | BL-driven ML forecasts (Upton & Hathaway 2014b) |

**Key insights / 핵심 통찰**:
1. Parker–Yoshimura sign rule is geometrically tight: equatorward butterfly needs either negative α at low latitudes or dominant tachocline shear.
2. Stochastic tilt scatter (σ~0.25) alone produces Grand-Minima-like epochs with exponential waiting times.
3. In advection-dominated BL models, **meridional-circulation speed sets the period** — contrast with mean-field αΩ.
4. Global MHD simulations can now self-consistently generate solar-like cycles (new in the 2020 edition vs. Paper #20).